# Phase 2 - DocLayNet layout detector smoke (T4)

Runner only (CLAUDE.md P1/P2). The Phase 2 **gate**: confirm `Aryn/deformable-detr-DocLayNet`
loads, exposes `config.id2label` with a Table class, detects a Table box, and that the box
crops - on a real T4 so runtime / VRAM are measured. It pins nothing; pinning
`config.LAYOUT_MODEL` is a manual step after the gate passes.

**Why transformers is pinned to 4.49.0:** the checkpoint was saved with transformers 4.36.2
and uses a timm resnet50 backbone. The v5 meta-init loader does not load that backbone's
weights, so detection collapses to degenerate boxes. Pinning below v5 restores the eager
loader that loads the backbone. Logic lives in `scripts/smoke_layout_detector.py`, not here.

In [ ]:
# Boot - clone-if-absent, pin the Phase 2 dev branch
import os
BRANCH = 'feature/phase2-doclaynet-layout'
if not os.path.isdir('/content/FinDocStructRAG/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
!cd /content/FinDocStructRAG && git fetch origin --quiet && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
%cd /content/FinDocStructRAG

In [ ]:
# Mount Drive so config.OUTPUT_ROOT (and config.LAYOUT_OUTPUT) resolve to Drive paths
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install - pin transformers below v5 so the eager loader loads the timm backbone.
# torch is preinstalled on Colab; the smoke subprocess picks up this pinned transformers.
!pip install -q "transformers==4.49.0" timm Pillow requests
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"

## Step 1 - confirm loading is fixed (example page)

Run the smoke on the model card's example page first. Loading is fixed when: (1) there is **no**
`Materializing param=` line, and (2) the top detections are **large, high-confidence** boxes -
not the top-center ~50px low-score specks that meant the backbone was unloaded. A Table is not
required here (the example page may not have one); this step only checks the model runs correctly.

In [ ]:
!python scripts/smoke_layout_detector.py --threshold 0.3

## Step 2 - the real gate (a page with a Table GT)

Only after Step 1 looks healthy. Pull one DocLayNet val page that has a Table annotation
(`category_id 9 == Table`), then run the smoke on it. The gate passes when a Table box is
detected and cropped (`smoke OK`); the crop is saved so it can be eyeballed.

In [ ]:
# Grab a DocLayNet val page containing a Table (category_id 9 == Table).
!pip install -q datasets
from datasets import load_dataset
from pathlib import Path

out = Path('/content/doclaynet_table_smoke.png')
ds = load_dataset('docling-project/DocLayNet-v1.1', split='val', streaming=True)
for i, ex in enumerate(ds):
    if 9 in ex['category_id']:
        ex['image'].convert('RGB').save(out)
        print('saved', out, '| idx', i, '| size', ex['image'].size, '| cats', set(ex['category_id']))
        break

In [ ]:
!python scripts/smoke_layout_detector.py --image /content/doclaynet_table_smoke.png --threshold 0.3 --save-crop /content/layout_table_crop.png

## Step 3 - factory integration: build_layout_detector + detect_layout + crop

Exercise the actual factory functions from `src/layout_detector.py` and
`src/table_detection.py`. Two sub-steps:

**3a (normal path)** — both detectors built; `detect_layout` runs primary first;
primary finds a 0.907 Table so fallback does NOT fire; table regions above
`MIN_CROP_SCORE` are cropped with the score-filter contract.

**3b (forced fallback)** — set `min_table_score=0.99` so the primary 0.907 table
doesn't qualify; TATR fallback fires and contributes its table; verifies the
sequential path and source labelling.

In [ ]:
# Step 3a - normal detect_layout path (primary wins; fallback should NOT fire)
import time
from PIL import Image
from src.layout_detector import build_layout_detector
from src.table_detection import build_table_transformer_detector
from src.layout_parsing import detect_layout, TABLE_LABEL
from src.bbox_utils import crop_with_padding
from src import config

img = Image.open('/content/doclaynet_table_smoke.png').convert('RGB')

t0 = time.time()
layout_det = build_layout_detector(config.LAYOUT_MODEL, threshold=0.3)
table_det  = build_table_transformer_detector(config.TATR_DETECTION_MODEL, threshold=0.5)
print(f"[load] both detectors ready in {time.time()-t0:.1f}s")

t0 = time.time()
regions = detect_layout(img, layout_det, table_det, min_table_score=0.5)
print(f"[detect] {len(regions)} regions in {time.time()-t0:.2f}s")
for r in sorted(regions, key=lambda r: -r.score):
    print(f"  {r.label:16s}  {r.score:.3f}  {[round(c) for c in r.box]}  [{r.source}]")

# Crop: score filter is the caller's responsibility (detect_layout returns candidates)
MIN_CROP_SCORE = 0.5
tables = [r for r in regions if r.label == TABLE_LABEL and r.score >= MIN_CROP_SCORE]
print(f"\n[crop] {len(tables)} table(s) >= {MIN_CROP_SCORE}")
for i, r in enumerate(tables):
    crop = crop_with_padding(img, r.box, pad=4)
    path = f'/content/factory_table_crop_{i}.png'
    crop.save(path)
    print(f"  crop {i}: box={[round(c) for c in r.box]}  size={crop.size}  saved {path}")

In [ ]:
# Step 3b - forced fallback path: set min_table_score=0.99 so primary 0.907 doesn't qualify;
# TATR fallback must fire and contribute its own table region
regions_fb = detect_layout(img, layout_det, table_det, min_table_score=0.99, dedup_iou=0.5)
tables_fb = [r for r in regions_fb if r.label == TABLE_LABEL]
sources    = {r.source for r in tables_fb}
print(f"[forced-fallback] {len(tables_fb)} table(s)  sources={sources}")
for r in tables_fb:
    print(f"  {r.label:16s}  {r.score:.3f}  [{r.source}]  box={[round(c) for c in r.box]}")
assert len(tables_fb) >= 1, "fallback must produce at least one table"
assert "table_fallback" in sources or "layout" in sources, "unexpected source"
print("fallback path OK")

## Step 4 - layout batch runner (debug subset, seed=7, n=20)

Run `scripts/run_layout_batch.py` on a fixed 20-page DocLayNet val subset.
Artifacts land on Drive under `config.LAYOUT_OUTPUT` (persistent across sessions):
- `regions/<page_id>.json` — full Region list per page
- `crops/<page_id>_table_<i>.png` — one PNG per table above `--table-threshold`
- `manifest.csv` — counters + `fallback_used` flag per page

Goal here: confirm artifact stream is stable and crop quality looks reasonable
before touching AP/IoU. Inspect the manifest summary and spot-check a few crops.

In [ ]:
!python scripts/run_layout_batch.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3 --table-threshold 0.5

In [ ]:
# Manifest preview + spot-check one crop
import pandas as pd
from pathlib import Path
from src import config
from IPython.display import display, Image as IPImage

manifest_path = config.LAYOUT_OUTPUT / "manifest.csv"
df = pd.read_csv(manifest_path)
print(df[["page_id", "status", "gt_tables", "num_regions", "num_tables", "num_cropped", "fallback_used"]].to_string(index=False))
print(f"\nprocessed={df['status'].eq('processed').sum()}  failed={df['status'].eq('failed').sum()}")
print(f"gt_table pages={df['gt_tables'].gt(0).sum()}  total crops={df['num_cropped'].sum()}  fallback pages={df['fallback_used'].sum()}")

# Display first crop found
crops_dir = config.LAYOUT_OUTPUT / "crops"
first_crop = next(crops_dir.glob("*.png"), None)
if first_crop:
    print(f"\nspot-check: {first_crop.name}")
    display(IPImage(str(first_crop), width=600))
else:
    print("no crops written")

## Step 5 - IoU diagnostic (Q1 / Q2 / Q3)

Re-runs detection on the same 20 pages capturing **pre-fallback** primary results
(so primary IoU is not contaminated by dedup removal) and computes per-page IoU
against DocLayNet GT annotations.

Answers:
- **Q1** — did primary miss the table entirely, or just score < 0.5?
- **Q2** — is fallback IoU actually better than primary on those pages?
- **Q3** — is table_threshold=0.5 too strict? (threshold sensitivity table)

Output: `outputs/layout/diagnostic.csv` on Drive.

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3 --table-threshold 0.5

In [ ]:
# Diagnostic CSV preview (sorted by best_iou_final desc)
import pandas as pd
from src import config

df = pd.read_csv(config.LAYOUT_OUTPUT / "diagnostic.csv")
cols = ["page_id", "gt_tables", "primary_tables", "primary_max_score",
        "fallback_used", "best_iou_primary", "best_iou_fallback", "best_iou_final"]
print(df[cols].sort_values("best_iou_final", ascending=False).to_string(index=False))

## Step 5b - False-positive diagnostic (pages with no GT table)

Run the same diagnostic on 20 pages that have **no** GT Table annotation.
Any crop produced here is a false positive. Goal: measure false-positive crop rate
and compare fallback behaviour on table-free pages.

Expected: `gt_tables=0` for all rows, `best_iou_*=0.0` throughout.
Watch: how many pages still get a crop? How often does fallback fire?

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --exclude-table-gt --primary-threshold 0.3 --table-threshold 0.5